In [4]:
import pandas as pd

# 1. 엑셀 파일 읽기 (pd.read_csv 대신 pd.read_excel 사용)
file_path = '카페_연습용_이미지_메시지_100개.xlsx'
df = pd.read_excel(file_path)

# 2. 데이터가 잘 들어왔는지, 컬럼명은 무엇인지 확인
print(df.head())
print("\n현재 컬럼명 목록:", df.columns)

  brand   place      shop_name        user        date  \
0  블루보틀   압구정카페    블루보틀 압구정 카페     가브리엘라70  2021-02-24   
1  블루보틀   압구정카페    블루보틀 압구정 카페  cbetrana03  2023-01-26   
2  블루보틀   여의도카페    블루보틀 여의도 카페        황대구리  2021-12-24   
3  블루보틀    역삼카페     블루보틀 역삼 카페         쪼그미  2020-01-12   
4  블루보틀  홍대팝업카페  블루보틀 홍대 팝업 카페     SKTOPIA  2023-07-08   

                           score  \
0                            NaN   
1                       커피가 맛있어요   
2  디저트가 맛있어요, 커피가 맛있어요, 음료가 맛있어요   
3                            NaN   
4                         뷰가 좋아요   

                                             content  \
0  #커피맛집 #테이블부족 #블루보틀 #블루보틀압구정점... 머그컵에 받았다가 테이블이...   
1                                            커피 맛있네요   
2  추천 메뉴 마셔봤는데 이색적(대추야자향)이고 맛있었어요~! 평일 오전에는 한적하니 좋네요   
3  커피는 맛있는데 사람이 너무 많고..장소에 비해서 자리가 너무 좁고 불편해서 오래 ...   
4  미국에서 자주 보았었던 블루보틀(Blue Bottle) 카페를 한국에서 처음으로 방...   

                                                 src  
0  https://search.pstatic.net/common/?auto

In [9]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import os

# 1. 데이터 로드
file_path = '카페_연습용_이미지_메시지_100개.xlsx'
df = pd.read_excel(file_path)

# 2. 저장할 폴더 설정
output_dir = './downloaded_images'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("🚀 1단계: 이미지 다운로드 및 형식 통일 시작...")

# 3. 'src' 컬럼의 URL을 순회하며 처리
success_count = 0
fail_count = 0

for idx, row in df.iterrows():
    # src 컬럼에 여러 링크가 쉼표(,)로 구분되어 있을 수 있으므로 분리
    urls = str(row['src']).split(',')
    
    for i, url in enumerate(urls):
        url = url.strip()
        try:
            # 1단계-1: 데이터 질 검사 (URL 접속 확인)
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                # 1단계-2: 파일 형식 통일 (PIL로 열어서 RGB 변환)
                img = Image.open(BytesIO(response.content))
                img = img.convert('RGB')
                
                # 파일명 규칙: 행번호_순번.jpg
                save_path = os.path.join(output_dir, f"cafe_{idx}_{i}.jpg")
                img.save(save_path, 'JPEG')
                success_count += 1
            else:
                print(f"⚠️ 접속 불가 (Status {response.status_code}): {url[:50]}...")
                fail_count += 1
                
        except Exception as e:
            # 1단계-3: 손상된 데이터 처리 (이미지가 깨졌거나 링크가 잘못된 경우)
            print(f"❌ 처리 실패: {url[:50]}... (에러: {e})")
            fail_count += 1

print("\n" + "="*30)
print(f"✅ 1단계 완료 보고")
print(f"- 성공적으로 다운로드 및 변환: {success_count}개")
print(f"- 링크 오류 및 손상 데이터: {fail_count}개")
print(f"- 저장 위치: {output_dir}")
print("="*30)

🚀 1단계: 이미지 다운로드 및 형식 통일 시작...

✅ 1단계 완료 보고
- 성공적으로 다운로드 및 변환: 139개
- 링크 오류 및 손상 데이터: 0개
- 저장 위치: ./downloaded_images


In [12]:
import os
import hashlib
import shutil

# 1. 경로 설정 (반드시 본인의 폴더명과 일치해야 합니다)
source_dir = './downloaded_images'  # 1단계에서 사진을 받은 폴더
unique_dir = './processed_2_unique' # 중복이 제거된 깨끗한 사진들이 담길 폴더

# 폴더가 없으면 새로 생성 (확실하게 생성하기 위해 os.makedirs 사용)
if not os.path.exists(unique_dir):
    os.makedirs(unique_dir)
    print(f"📂 새로운 폴더 생성 완료: {unique_dir}")

def calculate_hash(file_path):
    """파일의 고유값(MD5 해시)을 계산합니다."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        # 대용량 파일도 처리 가능하도록 조각(chunk) 단위로 읽기
        for chunk in iter(lambda: f.read(4096), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

# 2. 중복 확인 및 파일 복사
hash_dict = {}  # {해시값: 파일이름} 저장용
duplicate_count = 0

print("🔍 해시 함수 기반 중복 제거 프로세스 시작...")

# 원본 폴더 내의 이미지들을 하나씩 확인
file_list = [f for f in os.listdir(source_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for filename in file_list:
    file_path = os.path.join(source_dir, filename)
    
    # 1단계: 파일의 해시값 계산
    file_hash = calculate_hash(file_path)
    
    # 2단계: 중복 여부 판단
    if file_hash not in hash_dict:
        # 처음 등장한 이미지라면 '유니크' 폴더로 복사
        hash_dict[file_hash] = filename
        shutil.copy(file_path, os.path.join(unique_dir, filename))
    else:
        # 이미 존재하는 해시값이면 중복 데이터임
        print(f"📍 중복 파일 발견 및 제외: {filename} (원본: {hash_dict[file_hash]})")
        duplicate_count += 1

print("\n" + "="*40)
print(f"✅ 중복 제거 완료!")
print(f"- 원본 이미지 수: {len(file_list)}개")
print(f"- 제외된 중복 수: {duplicate_count}개")
print(f"- 최종 유니크 이미지: {len(hash_dict)}개 (저장위치: {unique_dir})")
print("="*40)

📂 새로운 폴더 생성 완료: ./processed_2_unique
🔍 해시 함수 기반 중복 제거 프로세스 시작...

✅ 중복 제거 완료!
- 원본 이미지 수: 139개
- 제외된 중복 수: 0개
- 최종 유니크 이미지: 139개 (저장위치: ./processed_2_unique)


In [14]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 18.0 MB/s  0:00:02 eta 0:00:01


In [15]:
import cv2
import os
import shutil

# 1. 경로 설정
source_dir = './processed_2_unique'    # 중복 제거된 유니크 이미지 폴더
quality_dir = './processed_2_high_quality' # 선명한 이미지 저장 폴더
blurry_dir = './processed_2_blurry'        # 흐린 이미지 저장 폴더

# 폴더 생성 및 확인
for d in [quality_dir, blurry_dir]:
    if not os.path.exists(d):
        os.makedirs(d)
        print(f"📂 폴더 생성됨: {d}")

# 2. 선명도 임계값 설정 (보통 100을 기준으로 하며, 데이터에 따라 조절합니다)
# 값이 클수록 더 엄격하게 필터링합니다 (더 선명한 사진만 통과).
THRESHOLD = 80.0

print(f"🚀 라플라시안 알고리즘 기반 질 검사 시작 (Threshold: {THRESHOLD})...")

high_quality_count = 0
blurry_count = 0

for filename in os.listdir(source_dir):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        file_path = os.path.join(source_dir, filename)
        
        # 이미지 불러오기
        image = cv2.imread(file_path)
        if image is None:
            continue
            
        # 그레이스케일 변환
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # 라플라시안 분산 값 계산 (선명도 점수)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        
        # 3. 점수에 따른 분류
        if laplacian_var < THRESHOLD:
            # 기준미달: 흐린 이미지로 분류
            shutil.copy(file_path, os.path.join(blurry_dir, filename))
            blurry_count += 1
        else:
            # 기준통과: 고품질 이미지로 분류
            shutil.copy(file_path, os.path.join(quality_dir, filename))
            high_quality_count += 1

print("\n" + "="*40)
print(f"✅ 질 검사 완료 보고")
print(f"- 고품질(통과): {high_quality_count}개")
print(f"- 저품질(흐림): {blurry_count}개")
print(f"📍 최종 결과 위치: {quality_dir}")
print("="*40)

📂 폴더 생성됨: ./processed_2_high_quality
📂 폴더 생성됨: ./processed_2_blurry
🚀 라플라시안 알고리즘 기반 질 검사 시작 (Threshold: 80.0)...

✅ 질 검사 완료 보고
- 고품질(통과): 131개
- 저품질(흐림): 8개
📍 최종 결과 위치: ./processed_2_high_quality


In [17]:
!pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 16.8 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 17.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.9/679.9 kB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision] [torchvision]


In [18]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.cluster import KMeans
from PIL import Image
import os
import shutil

# 1. 이미지 로드 및 특징 추출용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True).to(device)
model.eval() # 특징 추출 모드

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. 특징 벡터 추출 함수
def extract_features(img_path):
    img = Image.open(img_path).convert('RGB')
    img_t = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(img_t)
    return features.cpu().numpy().flatten()

# 3. 클러스터링 실행
source_dir = './processed_2_high_quality'
cluster_base_dir = './processed_2_clusters'
os.makedirs(cluster_base_dir, exist_ok=True)

img_files = [f for f in os.listdir(source_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
features_list = []
valid_files = []

print("🧠 이미지 특징 추출 중...")
for f in img_files:
    try:
        features_list.append(extract_features(os.path.join(source_dir, f)))
        valid_files.append(f)
    except: continue

# K-Means 그룹화 (K값은 데이터 양에 따라 조절, 여기서는 5개 그룹으로 설정)
K = 5 
kmeans = KMeans(n_clusters=K, random_state=42).fit(features_list)

# 4. 결과에 따라 폴더별로 이동
for i in range(K):
    os.makedirs(os.path.join(cluster_base_dir, f'cluster_{i}'), exist_ok=True)

for file, cluster_id in zip(valid_files, kmeans.labels_):
    shutil.copy(os.path.join(source_dir, file), os.path.join(cluster_base_dir, f'cluster_{cluster_id}', file))

print(f"✅ 클러스터링 완료! {K}개의 그룹으로 분류되었습니다.")

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/ohgyuwon/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:03<00:00, 15.4MB/s]


🧠 이미지 특징 추출 중...
✅ 클러스터링 완료! 5개의 그룹으로 분류되었습니다.
